Este notebook intenta debuguear el proceso de busqueda individual para nuevos perfumes, se intentó usar busqueda directa en google pero no se obtenian resultados porque las busquedas eran muy estrictas, para no hacer uso de expresiones regualares, tomaremos ventajas del buscador integrado de Fragrantica-

### IMPORTS Y VARIABLES

In [2]:
import sys
import os
import urllib.parse
import asyncio
import pandas as pd
from playwright.async_api import async_playwright

# Verificar versión de Python y directorio actual
print(f"Python executable: {sys.executable}")
print(f"Current working dir: {os.getcwd()}")

Python executable: /home/josuej/.pyenv/versions/3.11.11/bin/python
Current working dir: /home/josuej/Documentos/ProyectoIntegrador/debug


### INTENTAR UNA BUSQUEDA

In [7]:
import urllib.parse
from playwright.async_api import async_playwright

QUERY_PRUEBA = "FULL HOUSE"

async def test_search_fragrantica_exact_dom(query_text):
    encoded_query = urllib.parse.quote(query_text.lower())
    search_url = f"https://www.fragrantica.es/buscar/?query={encoded_query}"
    print(f"🔎 Navegando a la URL de búsqueda: {search_url}")

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled"]
        )
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
            locale="es-ES"
        )
        page = await context.new_page()

        try:
            # domcontentloaded no espera analíticas/anuncios en segundo plano (evita timeout)
            await page.goto(search_url, wait_until="domcontentloaded", timeout=12000)
            
            # Selector exacto basado en la jerarquía del DOM que compartiste
            grid_selector = 'div.ais-StateResults div.grid a[href*="/perfume/"]'
            
            # Esperar únicamente a que aparezca el grid de resultados
            await page.wait_for_selector(grid_selector, timeout=7000)
            
            # Tomar la primera coincidencia del grid
            first_card = page.locator(grid_selector).first
            
            if await first_card.count() > 0:
                href = await first_card.get_attribute("href")
                title = await first_card.get_attribute("title") or await first_card.get_attribute("aria-label") or "Perfume"
                
                if href:
                    if href.startswith("/"):
                        href = f"https://www.fragrantica.es{href}"
                    
                    clean_url = href.replace("https://www.fragrantica.com", "https://www.fragrantica.es")
                    
                    print(f"\n🎯 ¡COINCIDENCIA EXACTA OBTENIDA EN MENOS DE 2 SEGUNDOS!")
                    print(f"   Título: {title.strip()}")
                    print(f"   URL: {clean_url}")
                    
                    await browser.close()
                    return clean_url

            print("⚠️ No se encontraron elementos dentro del grid de resultados.")
            await browser.close()
            return None

        except Exception as e:
            print(f"❌ Error en la búsqueda: {e}")
            await browser.close()
            return None

# Ejecutar en la Celda 2 del notebook
url_resultado = await test_search_fragrantica_exact_dom(QUERY_PRUEBA)

🔎 Navegando a la URL de búsqueda: https://www.fragrantica.es/buscar/?query=full%20house

🎯 ¡COINCIDENCIA EXACTA OBTENIDA EN MENOS DE 2 SEGUNDOS!
   Título: Jo Milano Paris Game of Spades Full House unisex 2024
   URL: https://www.fragrantica.es/perfume/Jo-Milano-Paris/Game-of-Spades-Full-House-105095.html


#### SCRAPER SOBRE UN SOLO ELEMENTO

In [8]:
import subprocess

if url_resultado:
    # Ajusta la ruta al scraper según tu estructura
    scraper_path = os.path.abspath("../scraper/scraper_busqueda.py")
    print(f"🚀 Ejecutando subproceso en: {scraper_path}")
    
    cmd = [sys.executable, scraper_path, "--single-url", url_resultado]
    
    try:
        resultado = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=40
        )
        print("--- STDOUT (Salida del scraper) ---")
        print(resultado.stdout)
        
        if resultado.stderr:
            print("--- STDERR (Errores) ---")
            print(resultado.stderr)
            
    except subprocess.TimeoutExpired:
        print("⏰ El subproceso excedió el tiempo límite de 40 segundos.")
    except Exception as e:
        print(f"💥 Error al ejecutar subproceso: {e}")
else:
    print("Skipping scraper test because no URL was found.")

🚀 Ejecutando subproceso en: /home/josuej/Documentos/ProyectoIntegrador/scraper/scraper_busqueda.py
--- STDOUT (Salida del scraper) ---
🎯 Modo URL Individual activado para: https://www.fragrantica.es/perfume/Jo-Milano-Paris/Game-of-Spades-Full-House-105095.html
🔎 Navegando a: https://www.fragrantica.es/perfume/Jo-Milano-Paris/Game-of-Spades-Full-House-105095.html
✅ Extracción Exitosa -> Game of Spades Full House Jo Milano Paris para Hombres y Mujeres
📦 [1/1] Guardado en 7.46s.

🎉 Sesión guardada. Exitosos: 1 | Fallidos: 0



### VALIDAR QUE SE INCLUYE EN EL CSV DE RESULTADOS 

In [9]:
csv_path = os.path.abspath("../debug/fragrantica_data_from_scraper.csv")


if os.path.exists(csv_path):
    df = pd.read_csv(csv_path, sep='|')
    print(f"📊 Total de perfumes en CSV: {len(df)}")
    
    if url_resultado:
        match = df[df['url'] == url_resultado]
        if not match.empty:
            print("✅ El perfume recién extraído se guardó exitosamente en el CSV:")
            print(match[['name_raw', 'designer_raw', 'top_notes_raw']].to_dict(orient='records'))
        else:
            print("⚠️ La URL no se encuentra en el CSV.")
else:
    print(f"❌ No existe el archivo CSV en: {csv_path}")

📊 Total de perfumes en CSV: 2
✅ El perfume recién extraído se guardó exitosamente en el CSV:
[{'name_raw': 'Game of Spades Full House Jo Milano Paris para Hombres y Mujeres', 'designer_raw': 'Jo Milano Paris', 'top_notes_raw': nan}]


## EXTRAER MUESTRA DEL ARCHIVO PARQUET QUE YA CONTIENE LAS VARIABLES DE INGENIERIA

In [5]:
import pandas as pd
df = pd.read_parquet("../modelos/df_master.parquet")

# 2. Obtener una muestra aleatoria (ejemplo: 1,000 filas o el 10% del total)
muestra = df.sample(n=20, random_state=42)  # O usa frac=0.10 para el 10%
muestra.drop(columns=['reviews_text_corpus'], inplace=True, errors='ignore')

# 3. Guardar en CSV
muestra.to_csv("muestra.csv", index=False)